In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(q1_path)

In [ ]:
# Task 2: Write your code here:
print(df.shape)
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery TIme')
plt.ylabel('y')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
missing_vals = df.isnull().sum()
print(missing_vals)

# dropping where delivery time (target) is missing, can't predict without it

df_clean = df.dropna(subset=['Delivery_Time'])
missing_vals = df_clean.isnull().sum()
print(missing_vals)

# dropping Key features such as weather, traffic level, time of day and years of experience as they are important to predict actual times,
# and most cannot just be replaced with a mean, like weather
key_feats = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']
df_clean = df_clean.dropna(subset=key_feats)
missing_vals = df_clean.isnull().sum()
print(missing_vals)

# TODO try imputing/unknown

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)
print(df_clean.shape)
print(df_clean)

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

#onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_cols = df_clean.select_dtypes(include='object').columns

for col in cat_cols:
  le=LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
df_clean.head()
#X_copy = pd.DataFrame(onehot_encoder.fit_transform(X_copy), columns=cat_cols)


In [ ]:
# Task 5: Write your code here:  Apply feature scaling for all features (Use StandardScaler)
# scaling test data sadly

from sklearn.preprocessing import StandardScaler
import pandas

features = df_clean.columns.drop("Delivery_Time")

scaler = StandardScaler()

df_clean[features] = scaler.fit_transform(df_clean[features])

df_clean.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

import seaborn as sns
plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x='Delivery_Time')
plt.title('Distribution of Target Variable (Delivery Time)')
plt.xlabel('Time')
plt.ylabel('Count')
plt.show()

# There is an imbalance

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Deliver_Time']

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold, KFold
y_preds = []
skf = KFold(n_splits = 5, shuffle=True, random_state=42)
mae_scores = []
for i, (train_index, test_index) in enumerate(skf.split(X)):
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)
  y_fold_pred = model.predict(X_test)
  y_preds.append(y_fold_pred)

  mae_scores.append(mean_absolute_error(y_test, y_fold_pred))


mae_scores = np.array(mae_scores)
print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
plt.plot(mae_scores)

In [ ]:
# Calculate the baseline predictions (mean of the target)
from sklearn.metrics import mean_squared_error as sklearn_mae
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = sklearn_mae(y, baseline_pred)

print(f"Baseline MAE (using mean target): {baseline_mae:.4f}")

In [ ]:
# Task 1: Write your code here:

importances = {}

importances['Random Forest'] = model.feature_importances_
importances['Random Forest'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes = axes.flatten()
features = df_clean.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:



In [ ]:
# Task Bonus: Write your code here:
%pip install kagglehub catboost tqdm -q
from catboost import CatBoostClassifier

from sklearn.ensemble import VotingClassifier # I DONT HAVE ENOUGH TIME TO IMPLMENET THIS


model2 = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold, KFold
kf = KFold(n_splits = 5, shuffle=True, random_state=42)
mae_scores1 = []
mae_scores2 = []
final_mae = []
for i, (train_index, test_index) in enumerate(kf.split(X)):
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)
  y_fold_pred = model.predict(X_test)

  model2.fit(X_train, y_train)
  y_pred = model2.predict(X_test)



  mae1 = mean_absolute_error(y_test, y_fold_pred)
  mae2 = mean_absolute_error(y_test, y_pred)
  mae_scores1.append(mae1)
  mae_scores2.append(mae2)
  final_mae.append(mae1+mae2)



mae_scores1 = np.array(mae_scores1)
mae_scores2 = np.array(mae_scores2)
final_mae = np.array(final_mae)
print(f"5-Fold CV Results:")
print(f"MAE TREE:  ${mae_scores1.mean():,.2f}")
plt.plot(mae_scores1)
print(f"MAE CAT:  ${mae_scores2.mean():,.2f}")
plt.plot(mae_scores2)

print(f"MAE CAT:  ${mae_scores2.mean():,.2f}")
plt.plot(mae_scores2)

print(f"MAE BOTH:  ${final_mae.mean():,.2f}")
plt.plot(final_mae)